# dr-data-toolkit - End-to-End Demo

This notebook shows a simple end-to-end example using `dr_datakit`.

The goal is to demonstrate the toolkit step by step with synthetic data:

```text
create synthetic raw data -> load -> clean -> validate -> export parquet -> create Excel report
```

This notebook does not use confidential data, private paths, client-specific rules or proprietary templates.

## 1. Imports

We import pandas, pathlib and the main tools from `dr_datakit`.

In [ ]:
from pathlib import Path

import pandas as pd
from openpyxl import Workbook

from dr_datakit import (
    load_file,
    clean_df,
    validate_not_empty,
    validate_required_columns,
    validate_not_null,
    validate_unique,
    validate_non_negative,
    validate_allowed_values,
    export_csv,
    export_parquet,
    create_report,
)

## 2. Process date and paths

In a real notebook, the process date is usually defined manually.

The date controls which files are loaded and how outputs are named.

In [ ]:
process_date = pd.Timestamp("2026-01-31")
date_label = process_date.strftime("%Y%m%d")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"
REPORTS_DIR = OUTPUT_DIR / "reports"
TEMPLATES_DIR = BASE_DIR / "templates"

for folder in [RAW_DIR, PROCESSED_DIR, REPORTS_DIR, TEMPLATES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

raw_file = RAW_DIR / f"transactions_{date_label}.csv"
processed_file = PROCESSED_DIR / f"transactions_{date_label}.parquet"
template_file = TEMPLATES_DIR / "transactions_report_template.xlsx"
report_file = REPORTS_DIR / f"transactions_report_{date_label}.xlsx"

raw_file, processed_file, template_file, report_file

## 3. Create synthetic raw data

This section simulates a raw input file.

In a real process, this file would come from an operational system, folder, export or external source.

In [ ]:
df_raw_sample = pd.DataFrame(
    {
        "ID": ["T001", "T002", "T003", "T004", "T005"],
        "Client": ["Client A", "Client B", "Client A", "Client C", "Client B"],
        "Status": ["paid", "pending", "paid", "cancelled", "paid"],
        "Amount": ["1000,50", "2500,00", "1750,25", "500,00", "3200,75"],
        "Transaction Date": [
            "2026-01-03",
            "2026-01-10",
            "2026-01-15",
            "2026-01-20",
            "2026-01-28",
        ],
    }
)

df_raw_sample

## 4. Export synthetic data as raw CSV

The toolkit uses `export_csv()` to save the raw sample with standard defaults.

In [ ]:
export_csv(df_raw_sample, raw_file)

raw_file

## 5. Load raw data

Now we load the CSV file with `load_file()`.

By default, CSV files are loaded as strings. This helps preserve IDs, document numbers and raw formats before cleaning.

In [ ]:
df_raw = load_file(raw_file)

df_raw

## 6. Clean data

`clean_df()` applies basic cleaning rules:

- normalize column names
- strip strings
- drop fully empty rows
- convert selected numeric/date columns

In [ ]:
df = clean_df(
    df_raw,
    cols_float=["amount"],
    cols_date=["transaction_date"],
)

df

## 7. Validate data

Validation functions are designed to either pass silently or raise a clear error.

This keeps notebook execution easy to review.

In [ ]:
validate_not_empty(df, "transactions")

validate_required_columns(
    df,
    ["id", "client", "status", "amount", "transaction_date"],
)

validate_not_null(
    df,
    ["id", "client", "status", "amount"],
)

validate_unique(df, "id")

validate_non_negative(df, "amount")

validate_allowed_values(
    df,
    "status",
    ["paid", "pending", "cancelled"],
)

print("Validations passed.")

## 8. Export processed data to Parquet

Parquet is used as the processed dataset format because it preserves schema and is efficient to reuse.

In [ ]:
export_parquet(df, processed_file)

processed_file

## 9. Build report data

Each report can define its own summaries.

The toolkit does not force summary logic because every client, portfolio or report may require different outputs.

In [ ]:
df_summary = (
    df.groupby("status", as_index=False)
    .agg(
        transactions=("id", "count"),
        amount=("amount", "sum"),
    )
)

df_summary

In [ ]:
total_amount = df["amount"].sum()

report_values = [
    "Synthetic Transactions Report",
    process_date,
    total_amount,
]

report_values

## 10. Create a simple Excel template

In real use, the template may already exist and contain logos, formulas, fixed text and formatting.

Here we create a minimal template only for demonstration.

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = "Report"

ws["B2"] = "Transactions Report"
ws["B3"] = "Generated from synthetic data"

ws["B5"] = "Report name"
ws["B6"] = "Process date"
ws["B7"] = "Total amount"

ws["B10"] = "Summary"

wb.save(template_file)
wb.close()

template_file

## 11. Create the final Excel report

`create_report()` copies the template and writes data blocks into specific cells.

In this example:

- `report_values` is written next to the fixed labels.
- `df_summary` is written as an Excel table.

In [ ]:
create_report(
    template_path=template_file,
    output_path=report_file,
    items=[
        {
            "sheet": "Report",
            "cell": "C5",
            "data": report_values,
            "include_header": False,
        },
        {
            "sheet": "Report",
            "cell": "B11",
            "data": df_summary,
            "table_name": "SummaryTable",
            "number_cols": ["transactions", "amount"],
        },
    ],
)

report_file

## 12. Final result

The example created:

- a raw CSV file
- a processed Parquet file
- an Excel template
- a final Excel report

This demonstrates the core idea of `dr-data-toolkit`: use simple reusable tools while keeping the notebook clear and flexible.

In [ ]:
print("Raw file:       ", raw_file)
print("Processed file: ", processed_file)
print("Template file:  ", template_file)
print("Report file:    ", report_file)